# OBJECT DETECTION

![alt text](ref_images/YOLOv5-Youtube-Video-Output.gif)

# Object Detection

Object Detection is a computer vision task that identifies **what objects are present in an image and where they are located**.

Unlike image classification, object detection also provides the **location of each object** using bounding boxes.

## Difference Between Classification and Object Detection

Image Classification

Image
↓
Single label

Example:
Cat image → "Cat"


Object Detection

Image
↓
Detect multiple objects
↓
Draw bounding boxes

Example:
Image → Person, Car, Dog

## Bounding Box

A bounding box is a rectangle used to **localize an object in an image**.

It is usually represented by four values:

(x, y, width, height)

or

(xmin, ymin, xmax, ymax)

Example:

Top-left corner → (xmin, ymin)
Bottom-right corner → (xmax, ymax)

In [ ]:
import cv2
from ultralytics import YOLO
from IPython.display import display, clear_output
from PIL import Image

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(0)

while True:
    
    ret, frame = cap.read()
    
    if not ret:
        break
    results = model(frame, classes=[0])
    # results = model(frame)

    annotated = results[0].plot()

    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    clear_output(wait=True)
    display(Image.fromarray(annotated))

In [2]:
"""
Simple Flask app for YOLOv8 object detection (yolov8n.pt).
Upload an image to get detections with bounding boxes.
"""

import io
import os
from flask import Flask, request, render_template, send_file
from ultralytics import YOLO

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024  # 16 MB max upload

# Load model once at startup (notebook-safe: no __file__)
try:
    _base = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _base = os.getcwd()
MODEL_PATH = os.path.join(_base, "yolov8n.pt")
model = YOLO(MODEL_PATH)


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():
    if "image" not in request.files:
        return "No image uploaded", 400

    file = request.files["image"]
    if file.filename == "":
        return "No image selected", 400

    allowed = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    ext = os.path.splitext(file.filename)[1].lower()
    if ext not in allowed:
        return f"Invalid format. Use: {', '.join(allowed)}", 400

    image_bytes = file.read()
    # Decode to numpy array (YOLO does not accept raw bytes)
    import numpy as np
    import cv2
    nparr = np.frombuffer(image_bytes, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if image is None:
        return "Could not decode image", 400

    results = model.predict(
        source=image,
        conf=float(request.form.get("conf", 0.25)),
        save=False,
        verbose=False,
    )

    annotated = results[0].plot()  # BGR numpy array
    _, buf = cv2.imencode(".jpg", annotated)
    return send_file(
        io.BytesIO(buf.tobytes()),
        mimetype="image/jpeg",
        as_attachment=False,
        download_name="detection.jpg",
    )


# Run the server (starts when you run this cell; stop via Interrupt Kernel)
app.run(debug=False, host="0.0.0.0", port=5000, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.163.111.197:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Mar/2026 10:25:14] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Mar/2026 10:25:15] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [13/Mar/2026 10:25:23] "POST /predict HTTP/1.1" 200 -


# YOLO

![alt text](ref_images/Object-detection-with-the-YOLO-model-The-model-divides-the-image-into-a-grid-and-for.png)

Figure 2: The Model. Our system models detection as a regression problem. It divides the image into an S x S grid and for each
grid cell predicts B bounding boxes, confidence for those boxes,
and C class probabilities. These predictions are encoded as an
S x S x (B * 5 + C) tensor.

Input Image
      ↓

Resize (640 × 640)
      ↓

Convolutional Neural Network
      ↓

Feature Map (example: 7 × 7)
      ↓

Grid Cells
      ↓

Bounding Box Predictions
      ↓

Non-Maximum Suppression
      ↓

Final Detected Objects

# How YOLO Does Feature Fusion

YOLO usually uses architectures like:

FPN (Feature Pyramid Network)

PANet (Path Aggregation Network)

BiFPN (in some models)

Operations used:

Upsampling

Downsampling

Concatenation

Addition

# LabelImg for Object Detection Dataset

LabelImg is an open-source annotation tool used to create bounding box labels for object detection datasets.

It allows us to:

• Draw bounding boxes around objects  
• Assign class labels  
• Export annotations for training models

# Training YOLOv5s using Ultralytics

Steps:

1. Install Ultralytics
2. Prepare dataset structure
3. Create dataset.yaml
4. Train YOLOv5s model
5. Run detection

# !pip install ultralytics
https://www.ultralytics.com/

In [1]:
import os
import random
from pathlib import Path
import yaml

# -------- Dataset Path --------
dataset_path = r"object_detection_dataset"
images_path = os.path.join(dataset_path, "images")

# -------- Split Ratios --------
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# -------- Collect Images --------
image_ext = (".jpg", ".jpeg", ".png")
images = [f for f in os.listdir(images_path) if f.lower().endswith(image_ext)]

random.shuffle(images)

# -------- Split --------
n = len(images)
train_split = int(n * train_ratio)
val_split = int(n * (train_ratio + val_ratio))

train_imgs = images[:train_split]
val_imgs = images[train_split:val_split]
test_imgs = images[val_split:]

# -------- Write TXT files --------
def write_txt(filename, img_list):
    with open(os.path.join(dataset_path, filename), "w") as f:
        for img in img_list:
            path = os.path.join(images_path, img).replace("\\","/")
            f.write(path + "\n")

write_txt("train.txt", train_imgs)
write_txt("val.txt", val_imgs)
write_txt("test.txt", test_imgs)

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("Test:", len(test_imgs))

# -------- Create dataset.yaml --------
yaml_data = {
    'path': dataset_path.replace("\\","/"),
    'train': 'train.txt',
    'val': 'val.txt',
    'test': 'test.txt',
    'nc': 1,
    'names': ['e.coli']
}

with open(os.path.join(dataset_path, "dataset.yaml"), "w") as f:
    yaml.dump(yaml_data, f)

print("dataset.yaml created")

Train: 35
Val: 9
Test: 6
dataset.yaml created


In [2]:
import torch
from ultralytics import YOLO

print("GPU Available:", torch.cuda.is_available())

GPU Available: False


In [11]:
model = YOLO("yolov8n.pt")   # small and fast

In [9]:
print(model.trainer.args)

task=detect
mode=train
model=yolov8n.pt
data=dataset.yaml
epochs=20
time=None
patience=100
batch=16
imgsz=640
save=True
save_period=-1
cache=False
device=cpu
workers=0
project=None
name=ecoli_detector10
exist_ok=False
pretrained=True
optimizer=auto
verbose=True
seed=0
deterministic=True
single_cls=False
rect=False
cos_lr=False
close_mosaic=10
resume=False
amp=True
fraction=1.0
profile=False
freeze=None
multi_scale=0.0
compile=False
overlap_mask=True
mask_ratio=4
dropout=0.0
val=True
split=val
save_json=False
conf=None
iou=0.7
max_det=300
half=False
dnn=False
plots=True
end2end=None
source=None
vid_stride=1
stream_buffer=False
visualize=False
augment=False
agnostic_nms=False
classes=None
retina_masks=False
embed=None
show=False
save_frames=False
save_txt=False
save_conf=False
save_crop=False
show_labels=True
show_conf=True
show_boxes=True
line_width=None
format=torchscript
keras=False
optimize=False
int8=False
dynamic=False
simplify=True
opset=None
workspace=None
nms=False
lr0=0.01
lrf=

In [4]:
import yaml

with open("dataset.yaml", "r") as file:
    data = yaml.safe_load(file)

print(data)

{'names': ['e.coli'], 'nc': 1, 'path': 'D:/UET Mardan Lectures/AI Course Content/object_detection_01', 'train': 'object_detection_dataset/train.txt', 'val': 'object_detection_dataset/val.txt', 'test': 'object_detection_dataset/test.txt'}


In [12]:
results = model.train(
    data="dataset.yaml",
    epochs=2,
    imgsz=640,
    batch=16,
    name="ecoli_detector"
)

Ultralytics 8.4.21  Python-3.10.19 torch-2.10.0+cpu CPU (Intel Core(TM) i7-10870H 2.20GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ecoli_detector11, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective

- True positive (TP): An object and its location are correctly detected by the model.‍
- False positive (FP): The model made a detection, but it was incorrect.‍
- False negative (FN): An object that was actually present in the image, but the model failed to detect it.‍
- True negative (TN): True negatives occur when the model correctly identifies the absence of an object.

# Example: Calculating Precision, Recall, F1 Score

Suppose we have an object detection model that detects **cars** in images.

### Ground Truth (Actual Objects)
Total cars in dataset = **10**

### Model Predictions

| Prediction | Result |
|---|---|
| Correct detections | 7 |
| Wrong detections | 2 |
| Missed objects | 3 |

So we have:

TP = 7  
FP = 2  
FN = 3

---

# 1. Precision

Precision measures how many predicted objects are correct.

$$
Precision = \frac{TP}{TP + FP}
$$

$$
Precision = \frac{7}{7 + 2}
$$

$$
Precision = \frac{7}{9} = 0.78
$$

Precision = **78%**

---

# 2. Recall

Recall measures how many real objects were detected.

$$
Recall = \frac{TP}{TP + FN}
$$

$$
Recall = \frac{7}{7 + 3}
$$

$$
Recall = \frac{7}{10} = 0.70
$$

Recall = **70%**

---

# 3. F1 Score

F1 score balances precision and recall.

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

$$
F1 = 2 \times \frac{0.78 \times 0.70}{0.78 + 0.70}
$$

$$
F1 = 0.74
$$

F1 Score = **74%**

---

# 4. How to calculate AP (Simple Example)

**AP** = area under the **Precision–Recall curve** for one class. Here is a minimal example for the **Car** class.

**Ground truth:** 3 cars in the image.

**Detections** (sorted by confidence, high to low):

| Rank | Confidence | Correct? (TP/FP) |
|------|------------|------------------|
| 1    | 0.95       | TP               |
| 2    | 0.80       | TP               |
| 3    | 0.70       | FP               |
| 4    | 0.60       | TP               |

**Precision and Recall** at each step (treating each rank as a threshold):

| Step | Detections so far | TP so far | Precision | Recall |
|------|-------------------|-----------|-----------|--------|
| 1    | 1                 | 1         | 1/1 = 1.00 | 1/3 = 0.33 |
| 2    | 2                 | 2         | 2/2 = 1.00 | 2/3 = 0.67 |
| 3    | 3                 | 2         | 2/3 = 0.67 | 2/3 = 0.67 |
| 4    | 4                 | 3         | 3/4 = 0.75 | 3/3 = 1.00 |

**AP** = area under the Precision–Recall curve (plot Precision vs Recall and take the area). For this curve:

$$
AP_{Car} \approx 0.75
$$

*(In practice, AP is computed with a fixed rule, e.g. 11-point interpolation or the area under the continuous curve.)*

---

# 5. mAP (Simple Example)

Suppose we have **two classes** and we computed AP for each:

| Class | AP |
|---|---|
| Car | 0.75 |
| Person | 0.85 |

$$
mAP = \frac{AP_1 + AP_2}{2}
$$

$$
mAP = \frac{0.75 + 0.85}{2}
$$

$$
mAP = 0.80
$$

mAP = **80%**

---

# 6. Final Results

| Metric | Value |
|---|---|
| Precision | 0.78 |
| Recall | 0.70 |
| F1 Score | 0.74 |
| mAP | 0.80 |

# Evaluation Metrics in Object Detection

## 1. Precision (P)

Precision measures how many predicted objects are correct.

$$
Precision = \frac{TP}{TP + FP}
$$

Where:

- **TP** = True Positives (correct detections)
- **FP** = False Positives (incorrect detections)

---

## 2. Recall (R)

Recall measures how many real objects were detected.

$$
Recall = \frac{TP}{TP + FN}
$$

Where:

- **TP** = True Positives
- **FN** = False Negatives (missed objects)

---

## 3. F1 Score

F1 Score balances precision and recall.

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

---

## 4. Mean Average Precision (mAP)

First compute **Average Precision (AP)** for each class:

$$
AP = \int_0^1 P(r) \, dr
$$

Then compute the mean over all classes:

$$
mAP = \frac{1}{N}\sum_{i=1}^{N} AP_i
$$

Where:

- **N** = number of classes
- **AP$_i$** = Average Precision of class $i$

---

## Summary

| Metric     | Formula |
| ---------- | ------- |
| Precision  | $\frac{TP}{TP+FP}$ |
| Recall     | $\frac{TP}{TP+FN}$ |
| F1 Score   | $2 \times \frac{Precision \times Recall}{Precision + Recall}$ |
| mAP        | Mean of Average Precision across classes |


In [ ]:
model = YOLO("runs/detect/ecoli_detector9/weights/best.pt")

results = model.predict(
    source=r"D:/UET Mardan Lectures/AI Course Content/Week 04/object_detection_dataset/images",
    conf=0.25,
    save=True
)


image 1/50 D:\UET Mardan Lectures\AI Course Content\Week 04\object_detection_dataset\images\01_11_2024_24 hours_other bacteria 5_3.jpg: 384x640 (no detections), 95.5ms
image 2/50 D:\UET Mardan Lectures\AI Course Content\Week 04\object_detection_dataset\images\06_23_23_labelled_MLSB11_42_13H_4.jpg: 384x640 (no detections), 34.9ms
image 3/50 D:\UET Mardan Lectures\AI Course Content\Week 04\object_detection_dataset\images\06_23_23_labelled_MLSB11_42_8H_33.jpg: 384x640 (no detections), 38.3ms
image 4/50 D:\UET Mardan Lectures\AI Course Content\Week 04\object_detection_dataset\images\07_05_23_ecoli_11H_mlsb_datamlsb11H_9.jpg: 384x640 (no detections), 38.3ms
image 5/50 D:\UET Mardan Lectures\AI Course Content\Week 04\object_detection_dataset\images\10_13_23_10H_baragate_01_WIN_20231013_22_34_48_Pro.jpg: 384x640 (no detections), 44.3ms
image 6/50 D:\UET Mardan Lectures\AI Course Content\Week 04\object_detection_dataset\images\10_13_23_10H_baragte_03_WIN_20231013_22_29_15_Pro.jpg: 384x640 (no

In [14]:
model = YOLO(r"D:\office_projects_data\E.coli\E.coli_Yolov8_6k_images\runs\detect\train3\weights/best.pt")

results = model.predict(
    source=r"object_detection_dataset/images",
    conf=0.25,
    save=True
)


image 1/50 d:\UET Mardan Lectures\AI Course Content\object_detection_01\object_detection_dataset\images\01_11_2024_24 hours_other bacteria 5_3.jpg: 384x640 10 e.colis, 109.3ms
image 2/50 d:\UET Mardan Lectures\AI Course Content\object_detection_01\object_detection_dataset\images\06_23_23_labelled_MLSB11_42_13H_4.jpg: 384x640 1 e.coli, 91.2ms
image 3/50 d:\UET Mardan Lectures\AI Course Content\object_detection_01\object_detection_dataset\images\06_23_23_labelled_MLSB11_42_8H_33.jpg: 384x640 5 e.colis, 91.6ms
image 4/50 d:\UET Mardan Lectures\AI Course Content\object_detection_01\object_detection_dataset\images\07_05_23_ecoli_11H_mlsb_datamlsb11H_9.jpg: 384x640 9 e.colis, 84.8ms
image 5/50 d:\UET Mardan Lectures\AI Course Content\object_detection_01\object_detection_dataset\images\10_13_23_10H_baragate_01_WIN_20231013_22_34_48_Pro.jpg: 384x640 1 e.coli, 209.9ms
image 6/50 d:\UET Mardan Lectures\AI Course Content\object_detection_01\object_detection_dataset\images\10_13_23_10H_baragte_03

# Applications of YOLO
1. Autonomous Driving:
In autonomous driving, real-time object detection is crucial for identifying pedestrians, vehicles, and obstacles. YOLO's speed and accuracy make it an ideal choice for enhancing the safety and reliability of self-driving cars.

2. Surveillance and Security: 
YOLO is widely used in surveillance systems to detect suspicious activities and objects. Its real-time capabilities allow for immediate response to potential threats, improving overall security.

3. Robotics: 
Robots equipped with YOLO can efficiently navigate environments, recognize objects, and perform tasks that require object detection. This enhances their functionality in manufacturing, logistics, and service industries.

4. Healthcare: 
In healthcare, YOLO can assist in medical imaging by detecting abnormalities in X-rays, MRIs, and other scans. Its accuracy and speed can aid in early diagnosis and treatment planning.

5. Agriculture: 
YOLO is used in agriculture to monitor crop health, detect pests, and manage livestock. Its real-time analysis helps farmers make informed decisions to optimize yields and maintain crop quality.